# Toy Agent Comparison

This notebook now uses the actual repo agents and the repo evaluation path on `TopKEnv`.
Read the outputs here as a mechanism study, not as the main paper benchmark pipeline.

The main checks in this version are:
- masked vs unmasked partial-feedback SPO
- full vs partial vs phased comparisons
- flip-set and contested-arm diagnostics from the repo evaluation utilities
- completed multi-arm runs instead of the interrupted larger-arm cell

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from lets_plot import LetsPlot

from notebooks.toy_agent_comparison_helpers import (
    default_main_configs,
    default_structured_repair_configs,
    default_structured_scale_configs,
    plot_diagnostics,
    plot_crossing_trace,
    plot_prediction_profiles,
    plot_regret,
    print_regret_summary,
    run_crossing_trace_suite,
    run_scale_cases,
    run_suite,
)

LetsPlot.setup_html()
SMOOTH_WINDOW = 31


In [2]:
W_2ARM = np.array([[1.0], [1.5]], dtype=float)
main_configs = default_main_configs(include_mse=True, include_pfy=True)
structured_configs = default_structured_repair_configs()
display(pd.DataFrame([{'group': 'main', 'label': label, **cfg} for label, cfg in main_configs]))
display(pd.DataFrame([{'group': 'structured', 'label': label, **cfg} for label, cfg in structured_configs]))


,group,label,agent_name,full_feedback,mask_grad,phased
0,main,BayesSPO (full),BayesianSPO,True,True,NaN
1,main,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN
2,main,"BayesSPO (partial, unmasked)",BayesianSPO,False,False,NaN
3,main,BayesSPO-Phased (partial),BayesianSPO,False,True,True
4,main,BayesianMSE (full),BayesianMSE,True,False,NaN
5,main,BayesianMSE (partial),BayesianMSE,False,False,NaN
6,main,BayesianMSE-Phased (partial),BayesianMSE,False,False,True


,group,label,agent_name,full_feedback,mask_grad,phased,margin_explore,phase_length,phase_epochs,boundary_focus,boundary_scale,residual_strength,residual_scale_by_sigma
0,structured,BayesSPO (full),BayesianSPO,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,structured,BayesSPO-Phased (partial),BayesianSPO,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,structured,"BayesSPO (partial, margin explore)",StructuredRepairSPO,False,True,NaN,True,50.0,5.0,NaN,NaN,NaN,NaN
3,structured,BayesSPO-Phased (structured),StructuredRepairSPO,False,True,True,True,NaN,NaN,1.0,0.1,0.5,True


## 2-arm linear

This is the easy, well-specified case. If the bottleneck story is right, full feedback should dominate, but partial-feedback SPO should not necessarily collapse.

In [3]:
linear_rounds, linear_summary, linear_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='linear',
    capture_agents=True,
)
display(linear_summary)
print_regret_summary(linear_summary)
display(plot_regret(linear_rounds, '2-arm linear regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(linear_rounds, '2-arm linear diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(linear_artifacts, W_2ARM, cost_fn='linear'))


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.028519,0.060973,0.9968,0.0616,0.082149,0.062761,0.995190
1,BayesianMSE (full),0.028856,0.053668,0.9960,0.0640,0.061613,0.064012,0.993587
2,BayesSPO-Phased (partial),1.729937,1.068811,0.9912,0.0672,0.132831,0.083268,0.983968
3,BayesianMSE-Phased (partial),1.748081,1.068922,0.9868,0.1016,0.112207,0.084699,0.975150
4,"BayesSPO (partial, masked)",2.175411,1.142655,0.9832,0.0824,0.124346,0.084687,0.967936
5,BayesianMSE (partial),2.190109,1.149687,0.9804,0.1008,0.112865,0.086051,0.962325
6,"BayesSPO (partial, unmasked)",2.884705,1.342071,0.9492,0.4128,0.163324,0.085512,0.899800


BayesSPO (full)               : regret = 0.0 +/- 0.1
BayesianMSE (full)            : regret = 0.0 +/- 0.1
BayesSPO-Phased (partial)     : regret = 1.7 +/- 1.1
BayesianMSE-Phased (partial)  : regret = 1.7 +/- 1.1
BayesSPO (partial, masked)    : regret = 2.2 +/- 1.1
BayesianMSE (partial)         : regret = 2.2 +/- 1.1
BayesSPO (partial, unmasked)  : regret = 2.9 +/- 1.3


## 2-arm sigmoid

This is the misspecified case. The direct masked vs unmasked SPO comparison is the key check for the claim that the main failure mode is contaminated imputed comparisons rather than gradient masking alone.

In [4]:
sigmoid_rounds, sigmoid_summary, sigmoid_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(sigmoid_summary)
print_regret_summary(sigmoid_summary)
display(plot_regret(sigmoid_rounds, '2-arm sigmoid regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(sigmoid_rounds, '2-arm sigmoid diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(sigmoid_artifacts, W_2ARM, cost_fn='sigmoid'))


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.6408,0.493524,0.064343,0.992786
1,BayesianMSE (full),0.214252,0.205122,0.9508,0.6976,0.267097,0.084857,0.903006
2,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,1.4384,0.135753,0.109506,0.612024
3,BayesianMSE-Phased (partial),10.174635,1.154786,0.8136,1.5192,0.143701,0.109259,0.628056
4,"BayesSPO (partial, unmasked)",13.623184,0.812459,0.5844,1.9784,0.458430,0.085289,0.168737
5,BayesianMSE (partial),13.940120,1.124342,0.7128,1.9632,0.309359,0.104329,0.426052
6,"BayesSPO (partial, masked)",14.056453,1.309703,0.6920,1.9592,0.293107,0.106161,0.384369


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesianMSE (full)            : regret = 0.2 +/- 0.2
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
BayesianMSE-Phased (partial)  : regret = 10.2 +/- 1.2
BayesSPO (partial, unmasked)  : regret = 13.6 +/- 0.8
BayesianMSE (partial)         : regret = 13.9 +/- 1.1
BayesSPO (partial, masked)    : regret = 14.1 +/- 1.3


### Crossing Trace And Frozen Test Regret

This focuses on the SPO family only. The crossing trace tracks the estimated decision boundary over training, and the frozen test summary evaluates the final predictor greedily without UCB exploration.

In [5]:
spo_trace_configs = [item for item in main_configs if 'BayesSPO' in item[0]]
sigmoid_trace, sigmoid_frozen_summary, sigmoid_true_crossing = run_crossing_trace_suite(
    W_true=W_2ARM,
    k=1,
    configs=spo_trace_configs,
    n_rounds=500,
    hidden_dim=8,
    cost_fn='sigmoid',
    checkpoint_every=10,
)
display(sigmoid_frozen_summary)
display(plot_crossing_trace(sigmoid_trace, sigmoid_true_crossing, '2-arm sigmoid crossing trace'))


,agent,final_crossing,crossing_error,frozen_test_regret_mean,frozen_test_regret_std,frozen_decision_rate
0,BayesSPO (full),0.003047,0.003047,0.000000,0.000000,1.0000
1,BayesSPO-Phased (partial),-0.423427,0.423427,0.004332,0.011222,0.8310
2,"BayesSPO (partial, unmasked)",-0.458260,0.458260,0.004845,0.012146,0.8210
3,"BayesSPO (partial, masked)",-1.068735,1.068735,0.017758,0.028620,0.6445


### Structured Component Ablation

This isolates what the structured phased agent is actually composed of. The key comparison is not `margin explore` versus `structured` alone, because the latter is also phased and adds boundary-weighted replay plus residual target correction.

In [6]:
structured_rounds, structured_summary, structured_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=structured_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(structured_summary)
print_regret_summary(structured_summary)
display(plot_regret(structured_rounds, '2-arm sigmoid structured repair regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(structured_rounds, '2-arm sigmoid structured repair diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(structured_artifacts, W_2ARM, cost_fn='sigmoid'))


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.007834,0.014645,0.9956,0.6408,0.493524,0.064343,0.992786
1,BayesSPO-Phased (structured),4.780581,0.718324,0.7832,1.8056,0.134522,0.108475,0.567134
2,BayesSPO-Phased (partial),10.113685,1.380480,0.8056,1.4384,0.135753,0.109506,0.612024
3,"BayesSPO (partial, margin explore)",12.252804,1.017348,0.5716,1.9712,0.292700,0.106197,0.143086


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesSPO-Phased (structured)  : regret = 4.8 +/- 0.7
BayesSPO-Phased (partial)     : regret = 10.1 +/- 1.4
BayesSPO (partial, margin explore): regret = 12.3 +/- 1.0


In [7]:
structured_trace, structured_frozen_summary, structured_true_crossing = run_crossing_trace_suite(
    W_true=W_2ARM,
    k=1,
    configs=structured_configs,
    n_rounds=500,
    hidden_dim=8,
    cost_fn='sigmoid',
    checkpoint_every=10,
)
display(structured_frozen_summary)
display(plot_crossing_trace(structured_trace, structured_true_crossing, '2-arm sigmoid structured repair crossing trace'))


,agent,final_crossing,crossing_error,frozen_test_regret_mean,frozen_test_regret_std,frozen_decision_rate
0,BayesSPO (full),0.003047,0.003047,0.000000,0.000000,1.0000
1,"BayesSPO (partial, margin explore)",0.372481,0.372481,0.003200,0.009130,0.8565
2,BayesSPO-Phased (partial),-0.423427,0.423427,0.004332,0.011222,0.8310
3,BayesSPO-Phased (structured),0.647680,0.647680,0.008190,0.017583,0.7670


## Multi-arm Structured Scaling

These cases keep the smaller active shortlist and test whether the promising partial-feedback variants still help once the toy moves beyond 2 arms.

In [8]:
scale_configs = default_structured_scale_configs()
scale_cases = [
    dict(label='4 arms, k=2', n_arms=4, k=2, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=42),
    dict(label='8 arms, k=4', n_arms=8, k=4, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=43),
    dict(label='16 arms, k=4', n_arms=16, k=4, context_dim=5, hidden_dim=16, n_rounds=500, n_seeds=3, weight_seed=44),
]
scale_results = run_scale_cases(scale_cases, scale_configs, cost_fn='sigmoid')
for label, case in scale_results.items():
    print(label)
    display(case['summary'])
    print_regret_summary(case['summary'])
    print()


4 arms, k=2


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),1.797498,0.860302,0.840667,1.068000,0.487786,0.160804,0.886840
1,BayesSPO-Phased (structured),14.543525,2.926335,0.565333,2.694667,0.196663,0.219750,0.589552
2,BayesSPO-Phased (partial),16.329840,2.765846,0.563333,2.504000,0.191483,0.215374,0.600107
3,"BayesSPO (partial, margin explore)",35.830423,1.068412,0.260667,3.428000,0.348868,0.215997,0.248737


BayesSPO (full)               : regret = 1.8 +/- 0.9
BayesSPO-Phased (structured)  : regret = 14.5 +/- 2.9
BayesSPO-Phased (partial)     : regret = 16.3 +/- 2.8
BayesSPO (partial, margin explore): regret = 35.8 +/- 1.1

8 arms, k=4


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),6.593660,0.671865,0.554000,2.777333,0.489827,0.161445,0.860992
1,BayesSPO-Phased (structured),33.672414,2.165107,0.284667,5.241333,0.212199,0.219754,0.649855
2,BayesSPO-Phased (partial),35.280924,1.520341,0.305333,4.948000,0.205694,0.215870,0.668003
3,"BayesSPO (partial, margin explore)",82.192971,4.618845,0.051333,6.893333,0.375120,0.212490,0.283551


BayesSPO (full)               : regret = 6.6 +/- 0.7
BayesSPO-Phased (structured)  : regret = 33.7 +/- 2.2
BayesSPO-Phased (partial)     : regret = 35.3 +/- 1.5
BayesSPO (partial, margin explore): regret = 82.2 +/- 4.6

16 arms, k=4


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),6.854922,0.898933,0.490667,2.670667,0.495834,0.176085,0.931932
1,BayesSPO-Phased (structured),64.368057,4.022484,0.158667,6.696000,0.203515,0.356307,0.653077
2,BayesSPO-Phased (partial),77.357458,1.349647,0.172667,6.124000,0.193951,0.340522,0.691561
3,"BayesSPO (partial, margin explore)",168.662943,8.105709,0.005333,7.736000,0.382681,0.331172,0.250287


BayesSPO (full)               : regret = 6.9 +/- 0.9
BayesSPO-Phased (structured)  : regret = 64.4 +/- 4.0
BayesSPO-Phased (partial)     : regret = 77.4 +/- 1.3
BayesSPO (partial, margin explore): regret = 168.7 +/- 8.1



In [9]:
focus_cases = ['8 arms, k=4', '16 arms, k=4']
for focus_case in focus_cases:
    display(plot_regret(scale_results[focus_case]['rounds'], f'{focus_case} regret', smooth_window=SMOOTH_WINDOW))
    display(plot_diagnostics(scale_results[focus_case]['rounds'], f'{focus_case} diagnostics', smooth_window=SMOOTH_WINDOW))
